# Validation

## 1. Load

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import timm
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold

DATA_DIR = '/kaggle/input/competitions/cats-and-dogs'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
train['path'] = DATA_DIR + '/train/' + train.image_id + '.jpg'

label_cols = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
pet_cols = ['Moris', 'Motya', 'Biatrix']
print(len(train), label_cols)

12260 ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']


## 2. Groups (copies of one photo)

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model("resnet18.a1_in1k", pretrained=True, num_classes=0).eval().to(DEVICE)
tfm = timm.data.create_transform(**timm.data.resolve_data_config({}, model=model))

def load(path):
    im = Image.open(path)
    im.draft('RGB', (320, 320))
    return tfm(im.convert('RGB'))

feats = []
with ThreadPoolExecutor(os.cpu_count()) as ex, torch.no_grad():
    for s in range(0, len(train), 128):
        x = torch.stack(list(ex.map(load, train.path[s:s + 128])))
        feats.append(model(x.to(DEVICE)).cpu())
feats = torch.cat(feats).numpy()

THRESHOLD = 0.95
knn = NearestNeighbors(radius=1 - THRESHOLD, metric="cosine", algorithm="brute").fit(feats)
train["cluster"] = connected_components(knn.radius_neighbors_graph(feats), directed=False)[1]

nearest = 1 - knn.kneighbors(feats, n_neighbors=2)[0][:, 1]
for t in (0.90, 0.95, 0.98):
    print(f"images whose closest neighbour has similarity > {t}:", int((nearest > t).sum()))

print('images:', len(train), '| distinct photos:', train.cluster.nunique())

images whose closest neighbour has similarity > 0.9: 1244
images whose closest neighbour has similarity > 0.95: 148
images whose closest neighbour has similarity > 0.98: 18
images: 12260 | distinct photos: 12186


## 3. Stratum

In [3]:
pets_rarest_first = sorted(pet_cols, key=lambda p: train[p].sum())

def stratum(row):
    for p in pets_rarest_first:
        if row[p] == 1:
            return p
    return ('Cat' * row.Cat + 'Dog' * row.Dog) or 'None'

train['stratum'] = train.apply(stratum, axis=1)
train.stratum.value_counts()

stratum
Cat        4989
Dog        4984
None       2162
Motya        42
Moris        42
Biatrix      41
Name: count, dtype: int64

## 4. Folds

In [4]:
train['fold'] = -1
skf = StratifiedGroupKFold(5, shuffle=True, random_state=21)
for f, (_, val_idx) in enumerate(skf.split(train, train.stratum, groups=train.cluster)):
    train.loc[val_idx, 'fold'] = f

train[['image_id'] + label_cols + ['cluster', 'fold']].to_csv('folds.csv', index=False)

## 5. Check

In [5]:
cut = (train.groupby('cluster').fold.nunique() > 1).sum()
print('photos split across folds:', cut)

train.groupby('fold')[label_cols].sum().assign(images=train.groupby('fold').size())

photos split across folds: 0


,Cat,Dog,Moris,Motya,Biatrix,images
fold,,,,,,
0,1006,982,14,7,9,2448
1,1027,1007,12,7,7,2448
2,1024,1002,12,14,5,2452
3,1031,1003,7,4,14,2458
4,989,1031,11,10,6,2454
